## NbS river flood damages 

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import box
import matplotlib as mpl
from matplotlib.ticker import MultipleLocator, FuncFormatter
from matplotlib import font_manager as fm
import matplotlib.patheffects as pe
import re
import sys
lib_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
sys.path.append(str(lib_dir))
import Robyn_paper_2_defs
import sys, pathlib, importlib
sys.path.append(str(pathlib.Path("../../robyns_libraries").resolve()))
import Robyn_river_floods; importlib.reload(Robyn_river_floods)

### Base and output paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
mpl.rcParams.update(Robyn_paper_2_defs.NATURE_RC)
print("Using font:", Robyn_paper_2_defs.PREF)


MILLIONS = 1e-6

# SECTOR DAMAGE STATS

#### Read in damage files which underpin avoided damages by sector and subsector

In [ ]:
damage_future_path = base_path / "dphil_paper_2/processed_data/NbS_river_catchment/damage_future/damage__future.parquet"
damage_future = pd.read_parquet(damage_future_path)

MIN_ID = "10"  # ensemble_member value for the "min" scenario - min would be 10; max would be 7

# Ensure ensemble_member is comparable as string
damage_future["ensemble_member"] = damage_future["ensemble_member"].astype(str)

# --- Filter to MIN scenario only ----------------------------------------------
df_min = damage_future[damage_future["ensemble_member"] == MIN_ID].copy()

# # --- Compute avoided (baseline - future); create both column names you use ----
if "avoided__fluvial__ead" not in df_min.columns:
    required = {"baseline__fluvial__ead", "future__fluvial__ead"}
    if not required.issubset(df_min.columns):
        missing = required - set(df_min.columns)
        raise KeyError(f"Missing required columns: {missing}")
    df_min["avoided__fluvial__ead"] = (
        df_min["baseline__fluvial__ead"] - df_min["future__fluvial__ead"]
    )

# Alias for convenience/compatibility
df_min["avoided_ead"] = df_min["avoided__fluvial__ead"]

# --- Summary in USD millions --------------------------------------------------
total_avoided_usd_mn = df_min["avoided__fluvial__ead"].sum() * Robyn_river_floods.USD_PER_JMD * MILLIONS
print(f"MIN scenario (ensemble_member=10): total avoided EAD = {total_avoided_usd_mn:.2f} USD mn")

## Sector-level analysis 

In [ ]:

# === SECTOR — MIN SCENARIO ONLY (ensemble_member == "10") ===============
df = damage_future.loc[damage_future["ensemble_member"] == MIN_ID].copy()

# --- Sanity: required columns -------------------------------------------------
required_cols = {"asset_class", "baseline__fluvial__ead", "future__fluvial__ead"}
missing = required_cols - set(df.columns)

# Ensure numeric
df["baseline__fluvial__ead"] = pd.to_numeric(df["baseline__fluvial__ead"], errors="coerce")
df["future__fluvial__ead"]   = pd.to_numeric(df["future__fluvial__ead"], errors="coerce")

# Avoided (row-wise) in J$
df["avoided_ead"] = df["baseline__fluvial__ead"] - df["future__fluvial__ead"]

df["sector"] = df["asset_class"].map(Robyn_river_floods.sector_map)

# --- Aggregate by sector (keep NaN sector bucket if any) ----------------------
summary = (
    df.groupby("sector", dropna=False, as_index=False)
      .agg(
          baseline_ead=("baseline__fluvial__ead", "sum"),  # J$
          future_ead  =("future__fluvial__ead", "sum"),    # J$
          avoided_ead =("avoided_ead", "sum"),             # J$
      )
)

# Shares within each sector
summary = summary.assign(
    avoided_share = np.where(summary["baseline_ead"] > 0,
                             summary["avoided_ead"] / summary["baseline_ead"],
                             np.nan)
)

# Totals
baseline_total = float(summary["baseline_ead"].sum())
future_total   = float(summary["future_ead"].sum())
avoided_total  = float(summary["avoided_ead"].sum())

summary = summary.assign(
    share_of_total_baseline = np.where(baseline_total > 0, summary["baseline_ead"] / baseline_total, np.nan),
    share_of_total_avoided  = np.where(avoided_total  > 0, summary["avoided_ead"]  / avoided_total,  np.nan),
)

# TOTAL row
total_row = pd.DataFrame([{
    "sector": "TOTAL",
    "baseline_ead": baseline_total,
    "future_ead":   future_total,
    "avoided_ead":  avoided_total,
    "avoided_share": (avoided_total / baseline_total) if baseline_total > 0 else np.nan,
    "share_of_total_baseline": 1.0 if baseline_total > 0 else np.nan,
    "share_of_total_avoided":  1.0 if avoided_total  > 0 else np.nan,
}])

summary_with_total = pd.concat([summary, total_row], ignore_index=True)

# Sort by baseline desc, keep TOTAL last
is_total = summary_with_total["sector"].eq("TOTAL")
summary_with_total = pd.concat(
    [summary_with_total.loc[~is_total].sort_values("baseline_ead", ascending=False),
     summary_with_total.loc[ is_total]],
    ignore_index=True
)

# --- Unit-converted tables ----------------------------------------------------
# (a) J$ billions
sector_rollup_jmd_bil = summary_with_total.copy()
for c in ("baseline_ead", "future_ead", "avoided_ead"):
    sector_rollup_jmd_bil[c] = sector_rollup_jmd_bil[c] / 1e9

# (b) USD millions
sector_rollup_usd_mn = summary_with_total.copy()
for c in ("baseline_ead", "future_ead", "avoided_ead"):
    sector_rollup_usd_mn[c] = sector_rollup_usd_mn[c] * Robyn_river_floods.USD_PER_JMD / 1e6


print("— MIN scenario sector rollup (J$ billions) —")
Robyn_river_floods._pretty_print(sector_rollup_jmd_bil, "J$ bn")

print(f"— MIN scenario sector rollup (USD millions @ 1 USD = {Robyn_river_floods.JMD_PER_USD:.0f} J$) —")
Robyn_river_floods._pretty_print(sector_rollup_usd_mn, "USD mn")

# Sanity: TOTAL conversion consistency (J$ → USD mn)
tot_jmd = float(summary_with_total.loc[summary_with_total["sector"].eq("TOTAL"), "avoided_ead"])
tot_usd_mn = float(sector_rollup_usd_mn.loc[sector_rollup_usd_mn["sector"].eq("TOTAL"), "avoided_ead"])
print(f"TOTAL avoided_ead (MIN): J$ {tot_jmd:,.2f}  →  US$ {tot_usd_mn:,.2f} million (1 USD = {Robyn_river_floods.JMD_PER_USD:.0f} J$)")


In [ ]:
# === National EADs (USD millions) with legend below ==========================
Robyn_river_floods.money_cols = ["baseline_ead", "future_ead", "avoided_ead"]

# 0) Convert J$ → USD, then scale to millions
plot_usd = summary_with_total.copy()
for c in Robyn_river_floods.money_cols:
    plot_usd[c] = plot_usd[c] * Robyn_river_floods.USD_PER_JMD / Robyn_river_floods.SCALE_MN   # USD millions

# 1) Build plotting table (baseline & avoided only)
df = plot_usd[["sector", "baseline_ead", "avoided_ead"]].copy()
df = df[df["sector"].notna()]  # drop NaNs if any

# Put TOTAL at the end, sort others by baseline desc
order = (
    df.query('sector != "TOTAL"')
      .sort_values("baseline_ead", ascending=False)["sector"]
      .tolist()
    + ["TOTAL"]
)
df = df.set_index("sector").loc[order].reset_index()

# Values for stacked plotting
df["_baseline"] = df["baseline_ead"]
df["_avoided"]  = np.clip(df["avoided_ead"], 0, df["_baseline"])   # cap avoided ≤ baseline
df["_bottom"]   = df["_baseline"] - df["_avoided"]

x = np.arange(len(df))
is_total = df["sector"].eq("TOTAL").to_numpy()

# 2) Plot
fig, ax = plt.subplots(figsize=(8.5, 4.8))

# baseline bars (darker for TOTAL)
base_colors = np.where(is_total, "#B0B0B0", "#C8C8C8")
ax.bar(x, df["_baseline"], color=base_colors, edgecolor="#333333", linewidth=0.6, label="Baseline EAD")

# avoided portion as a hatched top segment
ax.bar(x, df["_avoided"], bottom=df["_bottom"],
       facecolor="none", edgecolor="#2E7D32", linewidth=0.9,
       hatch="///", label="Avoided through forest restoration", zorder=3)

# thin separator before TOTAL
if len(df) > 1:
    ax.axvline(len(df) - 1.5, linestyle=":", color="0.5", linewidth=0.8)

# y-grid (optional, subtle)
ax.grid(axis="y", linestyle=":", color="0.88", zorder=0)
ax.set_axisbelow(True)

# x/y labels & title
ax.set_xticks(x)
ax.set_xticklabels(df["sector"], rotation=0)
ax.set_ylabel("EADs (US$ million)")
ax.yaxis.set_major_locator(MultipleLocator(100))                 # or 50, etc.
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:,.0f}"))
ax.set_title("Expected Annual Damages (EADs) and % avoided through forest restoration",
             fontweight="bold")

# percent labels
ylim = ax.get_ylim()
offset = 0.01 * (ylim[1] - ylim[0])                 # small vertical offset in data units
threshold = 0.05 * df["_baseline"].max()            # 5% of max bar height

# sectors whose % label should be outside (above the bar)
force_outside = {"buildings", "transport", "TOTAL"}
ax.margins(y=0.06)  # a bit of top margin for outside labels

for i, r in df.iterrows():
    if r["_baseline"] <= 0 or r["_avoided"] <= 0:
        continue
    pct = r["_avoided"] / r["_baseline"]
    top = r["_bottom"] + r["_avoided"]

    want_outside = (r["sector"] in force_outside) or (r["_avoided"] < threshold)
    if want_outside:
        y, va = top + offset, "bottom"     # outside (above the bar)
    else:
        y, va = r["_bottom"] + r["_avoided"] / 2, "center"  # inside
    # ax.text(x[i], y, f"{pct:.0%}", ha="center", va=va, fontsize=9,
    #         color="#2E7D32", clip_on=False, zorder=5)
    ax.text(x[i], y, f"{pct:.1%}", ha="center", va=va, fontsize=9,
            color="#2E7D32", clip_on=False, zorder=5)

# legend BELOW the chart (centered)
ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.14),  # push below axes
    ncol=2,
    borderaxespad=0.0
)

# make room at the bottom for the legend
fig.subplots_adjust(bottom=0.25, right=0.98)

# ---- save BEFORE show ----

fname = output_dir / "national_avoided_EADs_usd_mn_min"
fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")

plt.show()
print("Saved to:", fname.with_suffix(".png"))
print("Assumed FX: 1 USD = 150 J$; values are in USD millions.")

# DAMAGES BY RETURN PERIOD (RP)

In [ ]:

# ===== Sector × RP summary — MIN scenario only (J$ bn + USD mn) ===============
# damage_future["ensemble_member"] = damage_future["ensemble_member"].astype(str)
damage_future_min = damage_future[damage_future["ensemble_member"] == MIN_ID].copy()

# --- 2) Ensure 'sector' exists (use your sector_map if needed) ----------------
# damage_future = damage_future.copy()
damage_future_min["sector"] = damage_future_min["asset_class"].map(Robyn_river_floods.sector_map)

# Optional: limit to the sectors you care about
wanted_sectors = ["buildings", "energy", "transport", "water"]
df = damage_future_min.loc[damage_future_min["sector"].isin(wanted_sectors)].copy()

# --- 3) Detect RP pairs present ----------------------------------------------
rp_levels = sorted(
    int(m.group(1))
    for col in df.columns
    if (m := re.match(r"baseline__fluvial__rp_(\d+)$", col))
    and f"future__fluvial__rp_{m.group(1)}" in df.columns
)
if not rp_levels:
    raise ValueError("No matching baseline/future RP columns found.")

# --- 4) Build long table across RPs ------------------------------------------
frames = []
for rp in rp_levels:
    bcol, fcol = f"baseline__fluvial__rp_{rp}", f"future__fluvial__rp_{rp}"
    tmp = (
        df[["sector", bcol, fcol]]
        .rename(columns={bcol: "baseline", fcol: "future"})
        .assign(rp=rp)
    )
    # be robust to non-numeric
    tmp["baseline"] = pd.to_numeric(tmp["baseline"], errors="coerce")
    tmp["future"]   = pd.to_numeric(tmp["future"],   errors="coerce")
    frames.append(tmp)

rp_long = pd.concat(frames, ignore_index=True)
rp_long["avoided"] = rp_long["baseline"] - rp_long["future"]

# --- 5) Aggregate: sector × RP and TOTAL per RP -------------------------------
sector_rp = (
    rp_long.groupby(["sector", "rp"], as_index=False)
           .agg(baseline=("baseline","sum"),
                future=("future","sum"),
                avoided=("avoided","sum"))
)
sector_rp["avoided_share"] = np.where(
    sector_rp["baseline"] > 0, sector_rp["avoided"] / sector_rp["baseline"], np.nan
)

totals = (
    rp_long.groupby("rp", as_index=False)
           .agg(baseline=("baseline","sum"),
                future=("future","sum"),
                avoided=("avoided","sum"))
)
totals["sector"] = "TOTAL"
totals["avoided_share"] = np.where(
    totals["baseline"] > 0, totals["avoided"] / totals["baseline"], np.nan
)

summary_rp = pd.concat([sector_rp, totals], ignore_index=True)

# --- 6) Sector shares of per-RP totals (exclude TOTAL from the ratios) -------
summary_rp["baseline_total_rp"] = summary_rp.groupby("rp")["baseline"].transform("sum")
summary_rp["avoided_total_rp"]  = summary_rp.groupby("rp")["avoided"].transform("sum")
is_total = summary_rp["sector"].eq("TOTAL")

summary_rp["share_of_total_baseline"] = np.where(
    (~is_total) & (summary_rp["baseline_total_rp"] > 0),
    summary_rp["baseline"] / summary_rp["baseline_total_rp"],
    np.nan
)
summary_rp["share_of_total_avoided"] = np.where(
    (~is_total) & (summary_rp["avoided_total_rp"] > 0),
    summary_rp["avoided"] / summary_rp["avoided_total_rp"],
    np.nan
)

# --- 7) Scaled copies: J$ billions and USD millions ---------------------------
Robyn_river_floods.money_cols = ["baseline", "future", "avoided"]

summary_rp_jmd_bil = summary_rp.copy()
for c in Robyn_river_floods.money_cols:
    summary_rp_jmd_bil[c] = summary_rp_jmd_bil[c] / 1e9  # J$ → J$ billions

summary_rp_usd_mn = summary_rp.copy()
for c in Robyn_river_floods.money_cols:
    summary_rp_usd_mn[c] = summary_rp_usd_mn[c] * Robyn_river_floods.USD_PER_JMD / 1e6  # J$ → USD millions

# --- 8) Pretty prints (1-decimal % as requested) ------------------------------
def _fmt_pct(v): return "" if pd.isna(v) else f"{v:.1%}"
def _fmt_money(v): return f"{v:,.2f}"


print("— Sector × RP (MIN scenario) — J$ billions —")
print(Robyn_river_floods._pretty(summary_rp_jmd_bil, "J$ bn").to_string(index=False))

print(f"— Sector × RP (MIN scenario) — USD millions @ 1 USD = {Robyn_river_floods.JMD_PER_USD:.0f} J$ —")
print(Robyn_river_floods._pretty(summary_rp_usd_mn, "USD mn").to_string(index=False))

# --- 9) Optional pivots for quick comparison ---------------------------------
avoided_jmd_pivot = (
    summary_rp_jmd_bil.pivot_table(index="sector", columns="rp", values="avoided", aggfunc="sum")
    .reindex(wanted_sectors + ["TOTAL"])
)
avoided_usd_pivot = (
    summary_rp_usd_mn.pivot_table(index="sector", columns="rp", values="avoided", aggfunc="sum")
    .reindex(wanted_sectors + ["TOTAL"])
)




print("Avoided damage by RP [J$ bn] (MIN):")
print(avoided_jmd_pivot.applymap(_fmt_money).to_string())
print("Avoided damage by RP [USD mn] (MIN):")
print(avoided_usd_pivot.applymap(_fmt_money).to_string())

In [ ]:
# --- 7) Scaled copies: J$ billions and USD millions ---------------------------
Robyn_river_floods.money_cols = ["baseline", "future", "avoided"]

summary_rp_jmd_bil = summary_rp.copy()
for c in Robyn_river_floods.money_cols:
    summary_rp_jmd_bil[c] = summary_rp_jmd_bil[c] / 1e9  # J$ → J$ billions

# Save long table (sector x RP)
summary_rp_jmd_bil.to_csv(output_dir / "sector_x_rp_min_JMD_bil_min.csv", index=False)

summary_rp_usd_mn = summary_rp.copy()
for c in Robyn_river_floods.money_cols:
    summary_rp_usd_mn[c] = summary_rp_usd_mn[c] * Robyn_river_floods.USD_PER_JMD / 1e6

summary_rp_usd_mn.to_csv(output_dir / "sector_x_rp_min_USD_mn_min.csv", index=False)



## Sub-sector analysis

In [ ]:
# === Subsector & sector summaries =========================

# Make sure inputs to avoided_ead are numeric
for col in ("baseline__fluvial__ead", "future__fluvial__ead"):
    if col not in damage_future_min.columns:
        raise KeyError(f"Missing required column: {col}")
    damage_future_min[col] = pd.to_numeric(damage_future_min[col], errors="coerce")

# Avoided in J$ (row-wise)
if "avoided_ead" not in damage_future_min.columns:
    damage_future_min["avoided_ead"] = (
        damage_future_min["baseline__fluvial__ead"] - damage_future_min["future__fluvial__ead"]
    )

# --- Build tables (MIN scenario dataframe) -----------------------------------
transport_with_total = Robyn_river_floods.build_subsector_summary(
    damage_future_min, Robyn_river_floods.transport_subsector_map, "transport_subsector", "TOTAL (transport)", unit="mn"
)
water_with_total = Robyn_river_floods.build_subsector_summary(
    damage_future_min, Robyn_river_floods.water_subsector_map, "water_subsector", "TOTAL (water)", unit="mn"
)
sector_with_total = Robyn_river_floods.build_sector_summary(damage_future_min)

# Rename share columns to be explicit (% of their sector totals)
transport_with_total = transport_with_total.rename(columns={"share_pct": "share_of_transport_pct"})
water_with_total     = water_with_total.rename(columns={"share_pct": "share_of_water_pct"})
transport_with_total["share_of_transport_pct"] = transport_with_total["share_of_transport_pct"].round(1)
water_with_total["share_of_water_pct"]         = water_with_total["share_of_water_pct"].round(1)

# Exclude TOTAL rows for category exports
transport = transport_with_total.query('transport_subsector != "TOTAL (transport)"').copy()
water = water_with_total.query('water_subsector != "TOTAL (water)"').copy()


transport[Robyn_river_floods.transport_cols].to_csv(output_dir / "avoided_ead_by_transport_subsector_JMD_USD_min.csv", index=False)
water[Robyn_river_floods.water_cols].to_csv(output_dir / "avoided_ead_by_water_subsector_JMD_USD_min.csv", index=False)

# Sector export
if sector_with_total is not None:
    sec = (sector_with_total.query('sector != "TOTAL"')
           .sort_values("avoided_ead_USD_bil", ascending=False)
           .copy())
    sec[[
        "sector", "avoided_ead_JMD", "avoided_ead_USD",
        "avoided_ead_JMD_bil", "avoided_ead_USD_bil",
    ]].to_csv(output_dir / "avoided_ead_by_sector_JMD_USD.csv", index=False)
    print(sec[["sector","avoided_ead_JMD_bil","avoided_ead_USD_bil"]].to_string(index=False))

print(transport[["transport_subsector","avoided_ead_JMD_mn","avoided_ead_USD_mn","share_of_transport_pct"]].to_string(index=False))
print(water[["water_subsector","avoided_ead_JMD_mn","avoided_ead_USD_mn","share_of_water_pct"]].to_string(index=False))